# TensorFlow/Keras vs. PyTorch dentro del deep learning

Recurso breve (2 celdas de código, 1 de markdown) que define el mismo ajuste — una secuencia sintética `y = x + 1` — primero con la API de bajo nivel de TensorFlow (`tf.Variable` + `GradientTape`) y después con PyTorch (`nn.Module` + `torch.optim`), para comparar el estilo de cada framework sobre el mismo problema.

| Aspecto | Estado |
|---|---|
| Profundidad | Comparación mínima de sintaxis entre frameworks, no un benchmark ni un análisis de performance |
| Dataset | Sintético (`x = 0..99`, `y = x+1`), sin relación con los otros notebooks del módulo |

In [1]:
import tensorflow as tf  # Importar TensorFlow para crear y entrenar el modelo de bajo nivel
import numpy as np  # Importar NumPy para manejar los datos numéricos
import time  # Importar el módulo time para medir el tiempo de entrenamiento
import pandas as pd  # Importar pandas para manejar los datos en un DataFrame
from bokeh.plotting import figure, output_notebook, show  # Importar funciones de Bokeh para crear y mostrar gráficos
from bokeh.models import ColumnDataSource, HoverTool  # Importar ColumnDataSource y HoverTool para manipular los datos y mejorar la interactividad del gráfico
from sklearn.metrics import mean_absolute_error  # Importar la métrica de error absoluto medio (MAE)

# Generar datos de ejemplo (secuencia simple)
x = np.array([i for i in range(100)])  # Datos de entrada: una secuencia de números del 0 al 99
y = np.array([i + 1 for i in range(100)])  # Datos de salida: el siguiente número en la secuencia

# Normalizar los datos
x_norm = x / float(max(x))  # Normalizar los datos de entrada dividiendo por el valor máximo
y_norm = y / float(max(y))  # Normalizar los datos de salida dividiendo por el valor máximo

# Separar en entrenamiento y prueba (80% entrenamiento, 20% prueba)
train_size = int(len(x) * 0.8)
x_train, x_test = x[:train_size], x[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Normalizar los datos de entrenamiento y prueba
x_train_norm = x_train / float(max(x))
y_train_norm = y_train / float(max(y))
x_test_norm = x_test / float(max(x))
y_test_norm = y_test / float(max(y))

# Convertir datos de entrenamiento a tensores para TensorFlow
x_train_tf = tf.convert_to_tensor(x_train_norm.reshape(-1, 1), dtype=tf.float32)  # Convertir x_train normalizado a tensor
y_train_tf = tf.convert_to_tensor(y_train_norm.reshape(-1, 1), dtype=tf.float32)  # Convertir y_train normalizado a tensor

# Definir el modelo con TensorFlow de bajo nivel
class SimpleNN(tf.Module):
    def __init__(self):
        self.W1 = tf.Variable(tf.random.normal([1, 50]), name='weight1')  # Inicializar pesos de la primera capa
        self.b1 = tf.Variable(tf.zeros([50]), name='bias1')  # Inicializar sesgo de la primera capa
        self.W2 = tf.Variable(tf.random.normal([50, 1]), name='weight2')  # Inicializar pesos de la segunda capa
        self.b2 = tf.Variable(tf.zeros([1]), name='bias2')  # Inicializar sesgo de la segunda capa

    def __call__(self, x):
        x = tf.matmul(x, self.W1) + self.b1  # Cálculo de salida de la primera capa
        x = tf.nn.relu(x)  # Aplicar función de activación ReLU
        x = tf.matmul(x, self.W2) + self.b2  # Cálculo de salida de la segunda capa
        return x

# Crear instancia de la red
model_tf = SimpleNN()

# Definir la función de pérdida
def loss_fn(model, x, y):
    y_pred = model(x)  # Predicción del modelo
    return tf.reduce_mean(tf.square(y_pred - y))  # Calcular el error cuadrático medio

# Definir el optimizador
optimizer = tf.optimizers.Adam(learning_rate=0.01)  # Inicializar el optimizador Adam con una tasa de aprendizaje de 0.01

# Función de entrenamiento
def train(model, x, y, epochs):
    losses = []  # Lista para almacenar las pérdidas
    for epoch in range(epochs):
        with tf.GradientTape() as tape:
            loss = loss_fn(model, x, y)  # Calcular la pérdida
        grads = tape.gradient(loss, [model.W1, model.b1, model.W2, model.b2])  # Calcular los gradientes
        optimizer.apply_gradients(zip(grads, [model.W1, model.b1, model.W2, model.b2]))  # Actualizar los pesos
        losses.append(loss.numpy())  # Almacenar la pérdida de cada época
    return losses

# Medir el tiempo de entrenamiento y obtener las pérdidas de TensorFlow
start_time_tf = time.time()  # Iniciar el temporizador
losses_tf = train(model_tf, x_train_tf, y_train_tf, epochs=200)  # Entrenar el modelo de TensorFlow
time_tf = time.time() - start_time_tf  # Calcular el tiempo total de entrenamiento

# Crear el modelo con Keras
from tensorflow.keras.models import Sequential  # Importar modelo secuencial de Keras
from tensorflow.keras.layers import Dense  # Importar capa densa de Keras

model_keras = Sequential([
    Dense(50, input_dim=1, activation='relu'),  # Definir capa oculta con 50 neuronas y activación ReLU
    Dense(1)  # Definir capa de salida
])

model_keras.compile(optimizer='adam', loss='mean_squared_error')  # Compilar el modelo usando el optimizador Adam y la pérdida de error cuadrático medio

# Medir el tiempo de entrenamiento y obtener las pérdidas de Keras
start_time_keras = time.time()  # Iniciar el temporizador
history_keras = model_keras.fit(x_train_norm, y_train_norm, epochs=200, verbose=0)  # Entrenar el modelo de Keras
time_keras = time.time() - start_time_keras  # Calcular el tiempo total de entrenamiento

# Crear un nuevo DataFrame para el rendimiento de los modelos
performance = pd.DataFrame({
    'Modelo': ['TensorFlow', 'Keras'],  # Nombres de los modelos
    'Tiempo de Entrenamiento (s)': [time_tf, time_keras],  # Tiempo total de entrenamiento para cada modelo
    'Perdida_Final': [losses_tf[-1], history_keras.history['loss'][-1]]  # Pérdida final después del entrenamiento
})

# Crear un ColumnDataSource con los datos de rendimiento
source = ColumnDataSource(performance)

# Asegurarse de que Bokeh renderice en el notebook
output_notebook()

# Crear gráfico de barras con Bokeh para comparar rendimiento
p_bar = figure(x_range=performance['Modelo'], title="Comparación de Rendimiento de Modelos",
               toolbar_location=None, tools="")  # Definir el gráfico de barras

# Añadir barras para el tiempo de entrenamiento
p_bar.vbar(x='Modelo', top='Tiempo de Entrenamiento (s)', width=0.4, source=source, 
           color="skyblue", legend_label="Tiempo de Entrenamiento (s)")

# Añadir barras para la pérdida final
p_bar.vbar(x='Modelo', top='Perdida_Final', width=0.2, source=source, 
           color="orange", legend_label="Pérdida Final", muted_alpha=0.2)

# Añadir herramientas de hover para mostrar detalles
p_bar.add_tools(HoverTool(tooltips=[
    ("Modelo", "@Modelo"), 
    ("Tiempo de Entrenamiento (s)", "@{Tiempo de Entrenamiento (s)}"), 
    ("Pérdida Final", "@Perdida_Final")
]))

# Configurar leyenda y etiquetas
p_bar.legend.visible = False  # Ocultar la leyenda
p_bar.y_range.start = 0
p_bar.xaxis.axis_label = "Modelos"
p_bar.yaxis.axis_label = "Valores"

# Mostrar el gráfico de barras
show(p_bar)

# Bloque extra: Evaluar y determinar el mejor modelo
if time_tf < time_keras and losses_tf[-1] < history_keras.history['loss'][-1]:
    mejor_modelo = 'TensorFlow'
    print("El mejor modelo es TensorFlow porque tiene un menor tiempo de entrenamiento y una menor pérdida final.")
elif time_keras < time_tf and history_keras.history['loss'][-1] < losses_tf[-1]:
    mejor_modelo = 'Keras'
    print("El mejor modelo es Keras porque tiene un menor tiempo de entrenamiento y una menor pérdida final.")
elif time_tf < time_keras:
    mejor_modelo = 'TensorFlow'
    print("El mejor modelo es TensorFlow porque tiene un menor tiempo de entrenamiento.")
else:
    mejor_modelo = 'Keras'
    print("El mejor modelo es Keras porque tiene un menor tiempo de entrenamiento.")

# Bloque extra: Evaluar el modelo con el conjunto de prueba
if mejor_modelo == 'TensorFlow':
    # Predecir con el conjunto de prueba normalizado
    predicciones_tf = model_tf(tf.convert_to_tensor(x_test_norm.reshape(-1, 1), dtype=tf.float32)).numpy()
    predicciones_desnormalizadas_tf = predicciones_tf * max(x)
    error_tf = mean_absolute_error(y_test, predicciones_desnormalizadas_tf)
    print(f"Error absoluto medio del modelo TensorFlow en el conjunto de prueba: {error_tf}")
else:
    # Predecir con el conjunto de prueba normalizado
    predicciones_keras = model_keras.predict(x_test_norm)
    predicciones_desnormalizadas_keras = predicciones_keras * max(x)
    error_keras = mean_absolute_error(y_test, predicciones_desnormalizadas_keras)
    print(f"Error absoluto medio del modelo Keras en el conjunto de prueba: {error_keras}")

# Bloque extra: Hacer una predicción con el mejor modelo
nuevo_dato = np.array([[7]])  # Nuevo dato de entrada no normalizado

# Normalizar el nuevo dato para la predicción
nuevo_dato_norm = nuevo_dato / float(max(x))

if mejor_modelo == 'TensorFlow':
    # Hacer predicción con el modelo de TensorFlow
    prediccion_tf = model_tf(tf.convert_to_tensor(nuevo_dato_norm, dtype=tf.float32))
    prediccion_desnormalizada = prediccion_tf.numpy()[0][0] * max(x)
    print(f"Predicción con el modelo TensorFlow para el nuevo dato {nuevo_dato[0][0]}: {prediccion_desnormalizada}")
else:
    # Convertir a NumPy array antes de predecir con Keras
    prediccion_keras = model_keras.predict(np.array(nuevo_dato_norm).reshape(-1, 1))
    prediccion_desnormalizada = prediccion_keras[0][0] * max(x)
    print(f"Predicción con el modelo Keras para el nuevo dato {nuevo_dato[0][0]}: {prediccion_desnormalizada}")

# Calcular el error de la predicción
valor_real = 8  # El siguiente número en la secuencia
error = mean_absolute_error([valor_real], [prediccion_desnormalizada])
print(f"Error absoluto medio de la predicción: {error}")


c:\Users\julio\AppData\Local\Programs\Python\Python39\lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Loading BokehJS ...

El mejor modelo es TensorFlow porque tiene un menor tiempo de entrenamiento.
Error absoluto medio del modelo TensorFlow en el conjunto de prueba: 1.1209884643554688
Predicción con el modelo TensorFlow para el nuevo dato 7: 8.018356829881668
Error absoluto medio de la predicción: 0.01835682988166809


Define y entrena el mismo ajuste con la API de bajo nivel de TensorFlow: variables (`tf.Variable`) y gradientes manuales con `GradientTape`, en vez de `Sequential`/`fit` como en los otros notebooks del módulo.

### Definición Pytorch
PyTorch es un Framework de Deep Learning de código abierto desarrollado por el grupo de investigación de IA de Facebook, compatible con Python y C++. Ofrece una interfaz más pulida en Python y se destaca por su flexibilidad al usar grafos dinámicos en lugar de estáticos. Este framework proporciona una codificación en alto nivel, cálculo automático de gradientes, paralelización de procesos y cuenta con el respaldo de una activa comunidad. A diferencia de otros, como TensorFlow, PyTorch utiliza conceptos básicos de Python, haciéndolo más intuitivo y simple para construir y entender modelos de aprendizaje profundo.

In [3]:
import torch
import torch.nn as nn #importar los modulos para construir redes neuronales 
import torch.optim as optim  # Importar optimizadores para entrenar el modelo
import numpy as np 
import time
from sklearn.metrics import mean_absolute_error  # Importar la métrica de error absoluto medio (MAE)

# Generar datos de ejemplo (secuencia simple)
x = np.array([i for i in range(100)])  # Datos de entrada: una secuencia de números del 0 al 99
y = np.array([i + 1 for i in range(100)])  # Datos de salida: el siguiente número en la secuencia

# Normalizar los datos
x_norm = x / float(max(x))  # Normalizar los datos de entrada dividiendo por el valor máximo
y_norm = y / float(max(y))  # Normalizar los datos de salida dividiendo por el valor máximo

# Separar en entrenamiento y prueba (80% entrenamiento, 20% prueba)
train_size = int(len(x) * 0.8)
x_train, x_test = x[:train_size], x[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

#convertir los datos en tensores pytorch 
x_train_tensor = torch.FloatTensor(x_train.reshape(-1, 1)) 
y_train_tensor = torch.FloatTensor(y_train.reshape(-1, 1)) 
x_test_tensor = torch.FloatTensor(x_test.reshape(-1, 1))
y_test_tensor = torch.FloatTensor(y_test.reshape(-1, 1)) 

# Definir la red neuronal profunda con PyTorch
class DeepNN(nn.Module):
    def __init__(self):
        super(DeepNN, self).__init__()
        self.fc1 = nn.Linear(1, 50)  # Capa oculta con 50 neuronas
        self.fc2 = nn.Linear(50, 50)  # Capa oculta con 50 neuronas
        self.fc3 = nn.Linear(50, 1)  # Capa de salida con 1 neurona

    def forward(self, x):
        x = torch.relu(self.fc1(x))  # Activación ReLU en la primera capa
        x = torch.relu(self.fc2(x))  # Activación ReLU en la segunda capa
        x = self.fc3(x)  # Capa de salida
        return x
    
#Instacianmos la red 
model = DeepNN()

# Definir la función de pérdida y el optimizador
criterion = nn.MSELoss()  # Función de pérdida de error cuadrático medio
optimizer = optim.Adam(model.parameters(), lr=0.01)  # Optimizador Adam con tasa de aprendizaje de 0.01

#Entrenar el modelo
start_time = time.time()  # Iniciar el temporizador
epochs = 200  # Número de épocas
losses = []  # Lista para almacenar las pérdidas

for epoch in range(epochs):
    model.train()  # Poner el modelo en modo de entrenamiento
    optimizer.zero_grad()  # Limpiar los gradientes
    outputs = model(x_train_tensor)  # Pasar los datos de entrenamiento por el modelo
    loss = criterion(outputs, y_train_tensor)  # Calcular la pérdida
    loss.backward()  # Retropropagación para calcular los gradientes
    optimizer.step()  # Actualizar los pesos
    losses.append(loss.item())  # Almacenar la pérdida de cada época

# Tiempo de entrenamiento
training_time = time.time() - start_time  # Calcular el tiempo total de entrenamiento
print(f"Tiempo de entrenamiento: {training_time:.4f} segundos")

# Evaluar el modelo en el conjunto de prueba
model.eval()  # Poner el modelo en modo de evaluación
with torch.no_grad():  # No calcular los gradientes
    predictions = model(x_test_tensor)  # Hacer predicciones en el conjunto de prueba
    predictions = predictions.numpy()  # Convertir las predicciones a numpy
    y_test_numpy = y_test_tensor.numpy()  # Convertir los datos de prueba a numpy
    error = mean_absolute_error(y_test_numpy, predictions)  # Calcular el error absoluto medio (MAE)


print(f"Error absoluto medio del modelo PyTorch en el conjunto de prueba: {error:.4f}")

# Realizar una predicción específica
nuevo_dato = np.array([[7]])  # Nuevo dato de entrada
nuevo_dato_tensor = torch.FloatTensor(nuevo_dato)  # Convertir a tensor de PyTorch
with torch.no_grad():  # No calcular los gradientes
    prediccion = model(nuevo_dato_tensor)  # Hacer la predicción
    prediccion_valor = prediccion.item()  # Obtener el valor de la predicción

print(f"Predicción con el modelo PyTorch para el nuevo dato {nuevo_dato[0][0]}: {prediccion_valor}")

# Calcular el error de la predicción específica
valor_real = 8  # El siguiente número en la secuencia
error_prediccion = abs(valor_real - prediccion_valor)  # Error absoluto de la predicción
print(f"Error absoluto de la predicción específica: {error_prediccion:.4f}")





Tiempo de entrenamiento: 0.7625 segundos
Error absoluto medio del modelo PyTorch en el conjunto de prueba: 0.0060
Predicción con el modelo PyTorch para el nuevo dato 7: 7.978333473205566
Error absoluto de la predicción específica: 0.0217


Misma idea que la celda de TensorFlow, ahora con PyTorch: clase `nn.Module`, optimizador de `torch.optim` y loop de entrenamiento explícito (forward, backward, `step`). No se detectaron errores en este recurso: es una comparación de sintaxis entre frameworks, sin matrices de confusión ni cálculos económicos que revisar.